# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/siddanger7/flyrank-ml-internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **CTR / Engagement Opportunity Scoring**. Data: the gated warehouse release (`FlyRank/internship-warehouse`, build v20260703), mid-panel month **2026-03** for features, **2026-04** for the outcome window. Every number below came from a query in this notebook — no invented figures.

> **Access note:** this notebook reads gated remote Parquet. It needs a Hugging Face **Read** token in the environment variable `HF_TOKEN` (or a Colab Secret of the same name). No token is ever written into this notebook — it is a public repo.

## 1. The contract, in plain words

1. **One row =** one content page × one day (`report_date × client_hash_id × content_hash_id`) — a *page-day*. For the review list I roll this up to one row per page per month (the decision grain).
2. **Table:** `fact_content_daily_performance`, `month=2026-03` partition (a mid-panel month, not the `_sample` — the sample is the final month and would peek at the outcome window).
3. **Time window:** features cover 2026-03-01 → 2026-03-31; the outcome is measured in **April 2026**, the next month.
4. **Label / proxy:** `declined_apr` — the page's CTR in April below its March CTR (a directional proxy for engagement decline, not a ground truth).
5. **Deliberately excluded:** rows with no GSC data (`gsc_data_available = FALSE`), pages with no position (`gsc_avg_position = 0`), the `_sample` table, and anything from the future — future numbers appear once, as the trap in section 4, never as a feature.

In [1]:
import os
import numpy as np
import pandas as pd
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    raise RuntimeError("Set HF_TOKEN (Hugging Face Read token) as an env var or Colab Secret.")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [TOKEN])

MARCH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"
print("duckdb connected; reading month=2026-03 and month=2026-04 partitions only.")

duckdb connected; reading month=2026-03 and month=2026-04 partitions only.


## 2. Three facts, proven with queries (month = 2026-03)

**Fact A — grain:** one row really is one page-day per client. The probe groups by the claimed unit and asks for any duplicate; an empty result means the grain holds.

**Fact B — slice count and date span:** my lane slice = rows with GSC data and a real position (`gsc_data_available IS TRUE AND gsc_avg_position > 0`).

**Fact C — availability with `IS TRUE`:** not every row has every measurement. I count how many rows survive each flag, because GA4 zeros are meaningful only where `ga4_data_available IS TRUE`.

In [2]:
# Fact A — grain probe: any duplicate page-day for a client?
dup = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("duplicate page-day rows:", len(dup))

duplicate page-day rows: 0


In [3]:
# Fact B — slice row count and date span
rows_b, lo_b, hi_b = con.execute(f"""
    SELECT COUNT(*), MIN(report_date), MAX(report_date)
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0
""").fetchone()
print(f"slice rows: {rows_b:,}  |  span: {lo_b} -> {hi_b}")

slice rows: 3,447,872  |  span: 2026-03-01 -> 2026-03-31


In [4]:
# Fact C — availability: filter with IS TRUE, count what survives
tot, gsc_ok, ga4_ok = con.execute(f"""
    SELECT
      COUNT(*),
      COUNT(*) FILTER (WHERE gsc_data_available IS TRUE),
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
    FROM {MARCH}
""").fetchone()
print(f"total March page-day rows: {tot:,}")
print(f"gsc_data_available IS TRUE: {gsc_ok:,}  ({gsc_ok/tot:.1%})")
print(f"ga4_data_available IS TRUE: {ga4_ok:,}  ({ga4_ok/tot:.1%})")

total March page-day rows: 9,841,378
gsc_data_available IS TRUE: 3,611,061  (36.7%)
ga4_data_available IS TRUE: 413,966  (4.2%)


## 3. Five features (page-level, from March)

Decision moment: **the instant the April review queue is built** — everything here is measurable by 2026-03-31 23:59, before any action. Every feature is an aggregate over the closed March window.

| Feature | What it is | Knowable at the decision moment because |
|---|---|---|
| `impressions_30d` | GSC impressions in March | the month is closed; trailing data only, no future row is touched |
| `ctr_march` | clicks / impressions in March | both are summed from the same closed window |
| `avg_pos_30d` | mean `gsc_avg_position` (rows with position > 0) | position was measured before any April action |
| `ga4_sessions_30d` | GA4 sessions in March | only counted where `ga4_data_available IS TRUE` — the flag is part of the feature definition |
| `sessions_ai_30d` | AI-referral sessions in March | trailing and sparse by design; used as a volume/interest signal, not a classifier |

The feature frame below is one row per page for every GSC-visible page in March.

In [5]:
# Build the page-level feature frame from the March partition
feat = con.execute(f"""
    SELECT
      client_hash_id, content_hash_id,
      SUM(gsc_impressions) AS impressions_30d,
      SUM(gsc_clicks) AS clicks_30d,
      AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_pos_30d,
      SUM(ga4_sessions) AS ga4_sessions_30d,
      MAX(ga4_data_available) AS ga4_avail_any,
      SUM(sessions_ai) AS sessions_ai_30d
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()
feat["ctr_march"] = feat["clicks_30d"] / feat["impressions_30d"].replace(0, np.nan)
feat["ga4_avail_any"] = feat["ga4_avail_any"].fillna(False).astype(bool)
print("feature rows (pages in March):", len(feat))
feat[["content_hash_id", "impressions_30d", "ctr_march", "avg_pos_30d", "ga4_sessions_30d", "sessions_ai_30d"]].head().round(3)

feature rows (pages in March): 176738


,content_hash_id,impressions_30d,ctr_march,avg_pos_30d,ga4_sessions_30d,sessions_ai_30d
0,content_72bbaf4f12c7a5dc,6.0,0.0,3.000,0.0,0.0
1,content_71a0fd14719165a7,10.0,0.0,4.333,0.0,0.0
2,content_b20d2c4d94f034a9,4.0,0.0,6.333,0.0,0.0
3,content_459d57f9097ed1a0,2.0,0.0,4.500,0.0,0.0
4,content_7ed1cb48f39397b9,3.0,0.0,19.333,0.0,0.0


## 4. The trap — one label-derived column, on purpose

The label is `declined_apr = ctr_apr < ctr_march`. It is *computed from* the April columns. If I quietly add `ctr_apr` — a column the label is derived from, i.e. the future — as a 'feature', the quick score should jump toward perfect. That is the leakage lesson from notebook 02, performed on real warehouse data: **a score that looks amazing while a future column is in the frame is worthless.**

I build the April outcome, join it to the March features, compute the label, train one cheap logistic regression with the five honest features, and one with `ctr_apr` added. Then I delete the leaked column and keep the honest number.

In [6]:
# April outcome -> label (CTR declined vs March)
apr = con.execute(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS clicks_apr,
           SUM(gsc_impressions) AS impressions_apr
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()
apr["ctr_apr"] = apr["clicks_apr"] / apr["impressions_apr"].replace(0, np.nan)

frame = feat.merge(apr, on=["client_hash_id", "content_hash_id"], how="inner")
frame = frame[frame["impressions_30d"] >= 100]      # minimum volume, keeps noise out
frame = frame[frame["ctr_march"] > 0]
frame["declined_apr"] = (frame["ctr_apr"] < frame["ctr_march"]).astype(int)
print("label frame rows:", len(frame), "| declined rate:", frame["declined_apr"].mean().round(3))

FEATS = ["impressions_30d", "ctr_march", "avg_pos_30d", "ga4_sessions_30d", "sessions_ai_30d"]
LEAK = ["ctr_apr"]  # the label-derived / future column

def quick_auc(columns):
    X = frame[columns].copy()
    X["log_impressions"] = np.log1p(X["impressions_30d"])
    X["log_ga4_sessions"] = np.log1p(X["ga4_sessions_30d"])
    X["log_sessions_ai"] = np.log1p(X["sessions_ai_30d"])
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    y = frame["declined_apr"].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(max_iter=5000).fit(sc.transform(Xtr), ytr)
    return roc_auc_score(yte, m.predict_proba(sc.transform(Xte))[:, 1])

honest_auc = quick_auc(FEATS)
leaked_auc = quick_auc(FEATS + LEAK)
print(f"AUROC, 5 honest features only:        {honest_auc:.3f}")
print(f"AUROC WITH the label-derived column:  {leaked_auc:.3f}   <- the trap")

label frame rows: 63510 | declined rate: 0.702


AUROC, 5 honest features only:        0.653
AUROC WITH the label-derived column:  1.000   <- the trap


In [7]:
# Delete the leaked column; the honest number is the one we keep.
frame.drop(columns=LEAK, inplace=True)
kept = quick_auc(FEATS)
print(f"Leaked column removed. Honest AUROC on the 5 features: {kept:.3f}")
print("The model that looks perfect (1.0) only does because it was reading April. "
      "The 0.65 figure is the real signal from March alone.")

Leaked column removed. Honest AUROC on the 5 features: 0.653
The model that looks perfect (1.0) only does because it was reading April. The 0.65 figure is the real signal from March alone.


## 5. One named limitation

**The panel is unbalanced.** A single calendar month mixes pages with very different history depths, and GA4 is available for only ~4% of March rows — so `ga4_sessions_30d` is zero-filled for the rest, and I cannot separate "no engagement" from "not tracked" without the flag. This slice is decision-support (it ranks candidates for a human reviewer), not a causal claim: a low-CTR page is a page to *look at first*, never a guarantee a refresh recovers it.

## Self-check

- [x] Contract in plain words: one row = page-day; table + month; window; label/proxy; deliberate exclusion
- [x] Exactly three verification queries on month=2026-03 with outputs (grain, count + span, availability via `IS TRUE`)
- [x] Five-feature frame, each with an "available when?" line
- [x] Deliberate-leak experiment shown (AUROC 1.0) and the leaked column deleted (honest 0.653 kept)
- [x] One named limitation
- [x] No client names, URLs, or private queries; token read from env var, never printed
- [x] Careful words: observed, measured, directional, decision-support